In [1]:
# ============================================================
# WHU 边缘监督消融实验 (Kaggle 优化版)
# EdgeBranch: lambda_edge = 0 vs 0.5
# ============================================================

import os
import random
import json
import zipfile
import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models
from PIL import Image
from tqdm.auto import tqdm

# ============================================================
# 1. Kaggle 路径配置与数据集检查
# ============================================================

# 尝试寻找 Kaggle 输入路径或解压文件
KAGGLE_INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

DATA_ROOT = "/kaggle/working/whu_data"

# 自动处理 Kaggle 上的 zip 包或解压目录
def prepare_dataset():
    if os.path.exists(DATA_ROOT):
        return
    
    # 查找是否有解压好的文件夹或 zip 文件
    zip_src = None
    for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
        if "train" in dirs and "val" in dirs:
            # 已经解压的情况
            global TRAIN_IMG_DIR, TRAIN_LBL_DIR, VAL_IMG_DIR, VAL_LBL_DIR
            TRAIN_IMG_DIR = os.path.join(root, "train", "image")
            TRAIN_LBL_DIR = os.path.join(root, "train", "label")
            VAL_IMG_DIR = os.path.join(root, "val", "image")
            VAL_LBL_DIR = os.path.join(root, "val", "label")
            return
        for f in files:
            if f.endswith("whu_data.zip") or f.endswith(".zip"):
                zip_src = os.path.join(root, f)

    if zip_src:
        print(f"正在解压数据集: {zip_src} 到 {DATA_ROOT}...")
        with zipfile.ZipFile(zip_src, 'r') as zip_ref:
            zip_ref.extractall(DATA_ROOT)
        print("解压完成！")

# 初始化默认路径
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, "train", "image")
TRAIN_LBL_DIR = os.path.join(DATA_ROOT, "train", "label")
VAL_IMG_DIR = os.path.join(DATA_ROOT, "val", "image")
VAL_LBL_DIR = os.path.join(DATA_ROOT, "val", "label")

prepare_dataset()

SAVE_DIR = os.path.join(WORKING_DIR, "whu_edge_supervision_ablation")
os.makedirs(SAVE_DIR, exist_ok=True)

SUPERVISED_EDGE_WEIGHT = 0.5
BATCH_SIZE = 8
EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_WORKERS = 0# 在 Kaggle 上支持多进程加速
SEED = 2026

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Train image:", TRAIN_IMG_DIR)
print("Train label:", TRAIN_LBL_DIR)
print("Val image:", VAL_IMG_DIR)
print("Val label:", VAL_LBL_DIR)
print("Save dir:", SAVE_DIR)


# ============================================================
# 2. 固定随机种子
# ============================================================

def set_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. 数据集
# ============================================================

class WHUDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir

        self.img_names = sorted([
            f for f in os.listdir(img_dir)
            if not f.startswith(".")
            and f.lower().endswith((".tif", ".tiff", ".png", ".jpg", ".jpeg"))
        ])

        if len(self.img_names) == 0:
            raise RuntimeError(f"没有找到图像文件: {img_dir}")

        self.img_paths = [os.path.join(img_dir, name) for name in self.img_names]
        self.lbl_paths = [os.path.join(lbl_dir, name) for name in self.img_names]

        for path in self.lbl_paths:
            if not os.path.exists(path):
                raise FileNotFoundError(f"找不到对应标签: {path}")

        self.image_transform = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, index):
        img_path = self.img_paths[index]
        lbl_path = self.lbl_paths[index]

        image = Image.open(img_path).convert("RGB")
        image = self.image_transform(image)

        label = Image.open(lbl_path)
        label = label.resize((256, 256), Image.NEAREST)

        label_np = np.array(label)
        label_np = (label_np > 0).astype(np.uint8)

        edge_np = cv2.Canny(
            label_np * 255,
            threshold1=100,
            threshold2=200
        ).astype(np.float32) / 255.0

        label_tensor = torch.from_numpy(label_np).long()
        edge_tensor = torch.from_numpy(edge_np).float().unsqueeze(0)

        return image, label_tensor, edge_tensor


# ============================================================
# 4. 模型
# ============================================================

class UpgradedEdgeEnhancedNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        resnet = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        self.encoder = nn.Sequential(
            *list(resnet.children())[:6]
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.edge_branch = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.fusion = nn.Sequential(
            nn.Conv2d(64 + 32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, num_classes, kernel_size=1)
        )

        self.edge_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        semantic_feature = self.encoder(x)
        semantic_feature = self.decoder(semantic_feature)

        edge_feature = self.edge_branch(x)
        edge_logits = self.edge_head(edge_feature)

        if semantic_feature.shape[2:] != edge_feature.shape[2:]:
            semantic_feature = F.interpolate(
                semantic_feature,
                size=edge_feature.shape[2:],
                mode="bilinear",
                align_corners=False
            )

        fused_feature = torch.cat([semantic_feature, edge_feature], dim=1)
        segmentation_logits = self.fusion(fused_feature)

        return segmentation_logits, edge_logits


# ============================================================
# 5. 验证指标
# ============================================================

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    tp, fp, fn = 0, 0, 0

    for images, labels, _ in tqdm(loader, desc="Validation", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        seg_logits, _ = model(images)
        predictions = torch.argmax(seg_logits, dim=1)

        pred_building = predictions == 1
        true_building = labels == 1

        tp += torch.logical_and(pred_building, true_building).sum().item()
        fp += torch.logical_and(pred_building, torch.logical_not(true_building)).sum().item()
        fn += torch.logical_and(torch.logical_not(pred_building), true_building).sum().item()

    eps = 1e-8
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)

    return {
        "IoU": iou,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }


# ============================================================
# 6. 单组实验
# ============================================================

def train_one_condition(condition_name, edge_loss_weight):
    print("\n" + "=" * 80)
    print(f"开始实验: {condition_name}")
    print(f"Edge loss weight: {edge_loss_weight}")
    print("=" * 80)

    set_seed(SEED)

    train_dataset = WHUDataset(TRAIN_IMG_DIR, TRAIN_LBL_DIR)
    val_dataset = WHUDataset(VAL_IMG_DIR, VAL_LBL_DIR)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0) # 优化 DataLoader 性能
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0)
    )

    print("Train samples:", len(train_dataset))
    print("Val samples:", len(val_dataset))

    model = UpgradedEdgeEnhancedNet(num_classes=2).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    criterion_seg = nn.CrossEntropyLoss()
    criterion_edge = nn.MSELoss()

    best_iou = -1.0
    best_epoch = -1

    checkpoint_path = os.path.join(SAVE_DIR, f"{condition_name}.pth")
    history = []

    for epoch in range(EPOCHS):
        model.train()

        total_loss, total_seg_loss, total_edge_loss = 0.0, 0.0, 0.0

        progress = tqdm(
            train_loader,
            desc=f"{condition_name} Epoch {epoch + 1}/{EPOCHS}"
        )

        for images, labels, edge_labels in progress:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            edge_labels = edge_labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            seg_logits, edge_logits = model(images)

            loss_seg = criterion_seg(seg_logits, labels)
            loss_edge = criterion_edge(edge_logits, edge_labels)

            loss = loss_seg + edge_loss_weight * loss_edge

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_seg_loss += loss_seg.item()
            total_edge_loss += loss_edge.item()

            progress.set_postfix({
                "loss": f"{loss.item():.4f}",
                "seg": f"{loss_seg.item():.4f}",
                "edge": f"{loss_edge.item():.4f}"
            })

        avg_loss = total_loss / len(train_loader)
        val_metrics = evaluate_model(model, val_loader)
        current_iou = val_metrics["IoU"]

        record = {
            "epoch": epoch + 1,
            "train_loss": avg_loss,
            "train_seg_loss": total_seg_loss / len(train_loader),
            "train_edge_loss": total_edge_loss / len(train_loader),
            "val_iou": val_metrics["IoU"],
            "val_precision": val_metrics["Precision"],
            "val_recall": val_metrics["Recall"],
            "val_f1": val_metrics["F1"]
        }

        history.append(record)

        print(
            f"[{condition_name}] Epoch {epoch + 1:03d} | "
            f"Train Loss {avg_loss:.5f} | "
            f"Val IoU {val_metrics['IoU']:.5f} | "
            f"Val F1 {val_metrics['F1']:.5f}"
        )

        if current_iou > best_iou:
            best_iou = current_iou
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            print(f"[SAVE] {checkpoint_path} | best IoU={best_iou:.5f}")

    history_path = os.path.join(SAVE_DIR, f"{condition_name}_history.json")
    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    print("\n实验完成")
    print("Condition:", condition_name)
    print("Best epoch:", best_epoch)
    print("Best val IoU:", best_iou)

    return checkpoint_path, history


# ============================================================
# 7. 运行对照实验
# ============================================================

if __name__ == "__main__":
    # A：无边缘监督
    ckpt_without_edge_supervision, _ = train_one_condition(
        condition_name="edge_without_supervision",
        edge_loss_weight=0.0
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # B：带边缘监督
    ckpt_with_edge_supervision, _ = train_one_condition(
        condition_name="edge_with_supervision",
        edge_loss_weight=SUPERVISED_EDGE_WEIGHT
    )

    print("\n" + "=" * 80)
    print("两组消融实验顺利完成")
    print("Without supervision:", ckpt_without_edge_supervision)
    print("With supervision:", ckpt_with_edge_supervision)
    print("=" * 80)

Device: cuda
Train image: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
Train label: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
Val image: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
Val label: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
Save dir: /kaggle/working/whu_edge_supervision_ablation

开始实验: edge_without_supervision
Edge loss weight: 0.0
Train samples: 4736
Val samples: 1036
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 181MB/s]


edge_without_supervision Epoch 1/50:   0%|          | 0/592 [00:00<?, ?it/s]

Validation:   0%|          | 0/130 [00:00<?, ?it/s]

[edge_without_supervision] Epoch 001 | Train Loss 0.23215 | Val IoU 0.74729 | Val F1 0.85537
[SAVE] /kaggle/working/whu_edge_supervision_ablation/edge_without_supervision.pth | best IoU=0.74729


...

Validation:   0%|          | 0/130 [00:00<?, ?it/s]

[edge_with_supervision] Epoch 049 | Train Loss 0.04993 | Val IoU 0.85619 | Val F1 0.92253


edge_with_supervision Epoch 50/50:   0%|          | 0/592 [00:00<?, ?it/s]

Validation:   0%|          | 0/130 [00:00<?, ?it/s]

[edge_with_supervision] Epoch 050 | Train Loss 0.05018 | Val IoU 0.85405 | Val F1 0.92128

实验完成
Condition: edge_with_supervision
Best epoch: 30
Best val IoU: 0.8597061415883674

两组消融实验顺利完成
Without supervision: /kaggle/working/whu_edge_supervision_ablation/edge_without_supervision.pth
With supervision: /kaggle/working/whu_edge_supervision_ablation/edge_with_supervision.pth
